Script for testing results on healthy data

In [21]:

import mne
import numpy as np
import pandas as pd
import os.path as op
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from tensorflow.keras import backend as K
import os
from models import run_kfold_training, CNN_model_contraction_og


gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
mne.set_log_level("CRITICAL")

global frq
global window 
global step 

## Training Model

In [ ]:
# Define CNN model inputs 
features_all = pd.read_pickle("training_features_18042026.pkl")


subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]
else:
    features_all_temp = features_all


muscle_groups = features_all_temp["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_zygo = features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy()
y_corr = features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()

indices = np.arange(len(X))
idx_train, idx_test, muscle_train, muscle_test = train_test_split(indices, muscle_groups,test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [ ]:
# call function for training
results_single_head_contraction = run_kfold_training(
    model_func=CNN_model_contraction_og,
    X_train_full=X_train_full,
    X=X,
    y=y,
    input_shape=input_shape,
    num_classes=num_classes,
    feature_num=feature_num,
    compile_kwargs={
        "optimizer": "adam",
        "loss": "sparse_categorical_crossentropy",
        "metrics": ["accuracy"]
    },
    n_splits=5,
    epoch_len=epoch_len)

model_contraction = results_single_head_contraction["models"][-1]
#model_history_contraction = model_contraction.fit(X_train_full, y_train_full, epochs=10, validation_data=(X_test, y_test)) #, callbacks=[early_stop])

## Pre-processing data 

In [10]:
global frq
global window 
global step 

In [25]:
to_keep=['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2','EOG1','EOG2','Corr', 'Zygo', 'Menton','Trigger']
eeg_ch= ['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2']
emg_ch= ['Corr', 'Zygo', 'Menton'] # Menton is chin EMG for sleep scoring
eog_ch= ['EOG1', 'EOG2']
trigger_ch= ['Trigger']

inter_trigger_length = 30
num_epochs = 60 # 60 epochs for each session 
subject_exclude = [] 

subject_list=  ['NL01SS','NL02IF','NL03JV','NL04NF','NL05WW','NL06DJ','RL01AN','RL02GC','RL03JG','RL04VF','RL05JP',
              'RL06FM','RL07BR','RL08AE','RL09PC','RL10TV', 'RL12JL','RL13MT','RL14LT','RL15CL',
              'RL17CA','RL19RS','RL20EM','RL21MB','RL22AC','RL23ET','RL24DD']



raw_path= "/Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/EDF_healthy_participants"

current_index=0
inter_trigger_length=10
window = 50 
step = 1 

In [40]:
# pre-processes data for each subject and returns df with session divided into 60 epochs w meta-data attached 
def pre_process_subjets(subject):
    global raw_path
    global frq

    subject_name = subject
    file= op.join(raw_path,'{}.edf'.format(subject_name))
    raw =  mne.io.read_raw_edf(file,preload=True)

    if 'Fp1/F3' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'Fp1/F3':'Fp1' ,'Fp2/F4':'Fp2'})
        
    if '36' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'36':'Corr' ,'37':'Zygo','38':'Menton'})

    if 'E1' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'E1':'EOG1' ,'E2':'EOG2'})
    
    if 'Corru' in raw.info['ch_names']:
        raw.rename_channels({'Corru': 'Corr'})


    for ch in raw.info['ch_names']:
        if ch not in to_keep:
            raw.drop_channels([ch]) # only keeps channel that contains the trigger (where a stimulus was presented )

    raw.set_channel_types(mapping={'Corr':'emg','Zygo':'emg','Menton':'emg','Trigger':'stim','EOG1':'eog','EOG2':'eog'})

    raw= raw.resample(sfreq=250)

    filter_params_emg = {'lpass': 100,'hpass': 10,'notches': [50]}
    raw.filter(l_freq=filter_params_emg['hpass'],h_freq=filter_params_emg['lpass'],picks=emg_ch)
    filter_params_eeg_eog = {'lpass': 70,'hpass': 0.3,'notches': [50]}
    raw.filter(l_freq=filter_params_eeg_eog['hpass'],h_freq=filter_params_eeg_eog['lpass'],picks=eeg_ch+eog_ch)
    raw.notch_filter(filter_params_eeg_eog['notches'], method='fft', picks=emg_ch+eeg_ch+eog_ch)

    frq=raw.info['sfreq']

    # create dataframe based on events 
    events_all= mne.find_events(raw) # Get all the events (triggers) in your EEG data
    events= mne.pick_events(events_all,include=[201,202]) # Pick events of interest (where a stimulus was presented)

    df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
    df_triggers.drop(columns=['dunno'],inplace=True)
    df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']


    # incorporating meta_data 
    cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Expected_Muscle"] # columns with relevant information from dataframe 
    trial_info = pd.read_csv("Trial_information_healthy_participants.csv", usecols=cols_inc) 
    expected_muscle = ["Corr", "Zygo"]


 
    epochs = mne.Epochs(raw, events, tmin=-0, tmax=9,baseline=None, detrend=0,
                    reject=None, preload=True, on_missing='warn')

   

    epochs.metadata = trial_info[(trial_info["Subject"] == subject.split(" ")[0]) 
                                & (trial_info["Trigger"].isin([201.0, 202.0]))]



    return epochs, df_triggers 

In [ ]:
def make_features_df(subject_epoch,subject,block):
    features = pd.DataFrame(
        index=range(num_epochs),
        columns=[
            "Subject",
            "Nap_ID"
            "Triggers_Order_Nap", # epochs 
            "Muscle Expected",
            "Num_Contractions_Zygo",
            "Num_Contractions_Corr",
            "Zygo", # processed EMG signal for epoch 
            "Corr"
        ]
        )   
    
    print(len(subject_epoch))
    epoch = 1
    for t in range(len(subject_epoch)): 
        # extract epoch  
        epoch_zygo = np.squeeze(subject_epoch[t].get_data(picks=['Zygo']))
        epoch_corr = np.squeeze(subject_epoch[t].get_data(picks=['Corr']))

        nap_id = (t // 60) + 1
        if t==59:
            epoch = 1 
        

        # fill dataframe 
        features.loc[t] = [
            subject, 
            nap_id,
            epoch,
            subject_epoch[t].metadata['Expected_Muscle'].iloc[0],
            subject_epoch.metadata.iloc[t]["Nb_Zygo"],
            subject_epoch.metadata.iloc[t]["Nb_Corr"],
            epoch_zygo,
            epoch_corr
        ]
        epoch += 1 
        print(nap_id,epoch)
    
    return features

In [53]:
# extracting features for classification 
i = 0 
excl = 0 # count which subjects had to be excluded 

features_results_mat = []

for root,dirs,files in os.walk(raw_path):
    for file in files:
        if "dpa" not in file and ".DS_Store" not in file: # so only take each subject once 
            subject = file.split(".")[0]
            print(subject)




            subject_epoch, block = pre_process_subjets(subject)
            i += 1
            
            # make features data frame for each subject
            features = make_features_df(subject_epoch,subject,block)
            features_results_mat.append(features)

           

print(f"{i} trials processed, {excl} trials excluded")

SS12NS Data
300
SS08KG Data


KeyboardInterrupt: 

In [ ]:
# loop over each subject 
# get features df 